In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler scikit-learn --quiet
import awswrangler as wr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET  = 'gold-lstm-forecast'
GOLD_PATH  = f's3://{S3_BUCKET}/gold/xauusd_daily/features/xauusd_features.parquet'
MODEL_PATH = f's3://{S3_BUCKET}/gold/xauusd_daily/features'

SEQ_LEN    = 60

FEATURES = ['close','return','ma7','ma14','ma30','ma60','volatility_7','momentum_7']
TARGET   = 'target'

print(f'SEQ_LEN  : {SEQ_LEN}')
print(f'Features : {FEATURES}')

In [ ]:
# Cell 2 · Load Gold Data
df = wr.s3.read_parquet(path=GOLD_PATH)
df = df.sort_values('date').reset_index(drop=True)
print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].min()} -> {df["date"].max()}')
print(f'Missing    : {df.isnull().sum().sum()}')

In [ ]:
# Cell 3 · Chronological Train-Test Split
# ทำไมต้อง chronological ไม่ใช่ random:
# ถ้า random split ข้อมูลอนาคตจะรั่วเข้าไปใน training (data leakage)
split_date = '2024-12-31'

train = df[df['date'] <= split_date].copy()
test  = df[df['date'] >  split_date].copy()

print('=== Train-Test Split ===')
print(f'  Train : {len(train):,} rows | {train["date"].min()} -> {train["date"].max()}')
print(f'  Test  : {len(test):,} rows  | {test["date"].min()} -> {test["date"].max()}')

In [ ]:
# Cell 4 · Scaling (fit on train ONLY — prevent data leakage)
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

feature_scaler.fit(train[FEATURES])
target_scaler.fit(train[[TARGET]])

X_train_scaled = feature_scaler.transform(train[FEATURES])
X_test_scaled  = feature_scaler.transform(test[FEATURES])
y_train_scaled = target_scaler.transform(train[[TARGET]]).ravel()
y_test_scaled  = target_scaler.transform(test[[TARGET]]).ravel()

print('Scaling done')
print(f'  X_train range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]')
print(f'  X_test range : [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]')

In [ ]:
# Cell 5 · Sliding Window Sequence
def create_sequences(X, y, seq_len):
    # Window includes day i itself (the most recent known day) when predicting
    # y[i], which is next-day close relative to day i. Excluding day i would
    # discard the freshest, most informative observation for no reason.
    Xs, ys = [], []
    for i in range(seq_len - 1, len(X)):
        Xs.append(X[i-seq_len+1:i+1])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, SEQ_LEN)
X_test,  y_test  = create_sequences(X_test_scaled,  y_test_scaled,  SEQ_LEN)

print('=== Sequence Shape ===')
print(f'  X_train : {X_train.shape}  (samples, timesteps, features)')
print(f'  y_train : {y_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_test  : {y_test.shape}')

In [ ]:
# Cell 6 · Save to /tmp and S3
np.save('/tmp/X_train.npy', X_train)
np.save('/tmp/X_test.npy',  X_test)
np.save('/tmp/y_train.npy', y_train)
np.save('/tmp/y_test.npy',  y_test)

with open('/tmp/feature_scaler.pkl', 'wb') as f:
    pickle.dump(feature_scaler, f)
with open('/tmp/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

for fname in ['X_train.npy','X_test.npy','y_train.npy','y_test.npy',
              'feature_scaler.pkl','target_scaler.pkl']:
    wr.s3.upload(local_file=f'/tmp/{fname}', path=f'{MODEL_PATH}/{fname}')
    print(f'  Uploaded: {fname}')

# y_test[0] is the target for the window ending at test row (SEQ_LEN-1) — see
# create_sequences, which starts at i=seq_len-1, not seq_len. test_dates has to
# start at the same row or its dates run one day ahead of the predictions
# they're meant to label.
test_dates = test['date'].iloc[SEQ_LEN-1:].reset_index(drop=True)
test_dates.to_csv('/tmp/test_dates.csv', index=False)
wr.s3.upload(local_file='/tmp/test_dates.csv', path=f'{MODEL_PATH}/test_dates.csv')
print('  Uploaded: test_dates.csv')
print()
print('Notebook 04 DONE')
print('Next -> 05_LSTM_Training.ipynb')